 https://huggingface.co/Doha000/distilbert-medical-classifier

In [1]:
# ==================== CELL 2: Imports & Setup ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Create model directory (team standard)
os.makedirs('../models/classifier', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)

print("✅ Libraries loaded | Device:", "GPU" if torch.cuda.is_available() else "CPU")

✅ Libraries loaded | Device: CPU


In [3]:
# ==================== CELL 3: Load & Prepare Data ====================
df = pd.read_csv('data/processed/pubmedqa_cleaned_Labeled.csv')

# === RICHER INPUT: question + context (this alone gives big boost) ===
df['text'] = df['question'].astype(str) + " [SEP] " + df['context'].astype(str)

print(f"Dataset shape: {df.shape}")
print("\nLabel distribution:")
print(df['label'].value_counts())

# Label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label_id'] = df['label'].map(label2id)

# Compute class weights (this is the most important fix for imbalance)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(list(label2id.values())),
    y=df['label_id'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cuda' if torch.cuda.is_available() else 'cpu')

print(f"\n✅ Class weights computed: {class_weights}")
print(f"Number of classes: {len(label2id)}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/pubmedqa_cleaned_Labeled.csv'

In [ ]:
# ==================== CELL 4: Train / Validation / Test Split ====================
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label_id'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
# ==================== CELL 5: Custom Dataset (IMPROVED) ====================
class MedicalQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe['text'].tolist()          # ← Now using question + context
        self.labels = dataframe['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
# ==================== CELL 6: Create Datasets & DataLoaders ====================
train_dataset = MedicalQADataset(train_df, tokenizer)
val_dataset = MedicalQADataset(val_df, tokenizer)
test_dataset = MedicalQADataset(test_df, tokenizer)

print("✅ Datasets created successfully")

In [ ]:
# ==================== CELL 7: Model Setup ====================
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print("✅ DistilBERT model initialized with 6 output classes")

In [ ]:
# ==================== CELL 8: Training Arguments + Weighted Loss (BEST VERSION) ====================
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='../models/classifier/checkpoints',
    num_train_epochs=6,                    # Increased for better convergence
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,                    # Lower LR = more stable
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    label_smoothing_factor=0.1,            # Helps with imbalance
)

# Weighted compute_metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'f1': f1}

print("✅ Training arguments + class weights ready (6 epochs)")

In [ ]:
# ==================== CELL 9: Trainer Setup & Training (WEIGHTED) ====================
# Re-initialize model with class weights
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Apply class weights to loss function
model.config.problem_type = "single_label_classification"
model.class_weights = class_weights

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting IMPROVED fine-tuning (with class weights + rich input)...")
trainer.train()

# Save model
model.save_pretrained('../models/classifier/distilbert_classifier')
tokenizer.save_pretrained('../models/classifier/distilbert_classifier')
print("✅ Model saved successfully")

In [ ]:
# ==================== CELL 10: Evaluation on Test Set ====================
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print("=== Classification Report (Test Set) ===")
print(classification_report(test_df['label_id'], preds, target_names=list(label2id.keys())))

# Confusion Matrix
cm = confusion_matrix(test_df['label_id'], preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(label2id.keys()),
            yticklabels=list(label2id.keys()))
plt.title('Confusion Matrix - DistilBERT Classifier')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('../reports/figures/06_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== CELL 11: Final KPI Check & Summary ====================
f1 = f1_score(test_df['label_id'], preds, average='weighted')
print(f"\n🎯 Weighted F1-Score on Test Set: {f1:.4f}")

if f1 >= 0.78:
    print("✅ Target achieved! F1 ≥ 78%")
else:
    print("⚠️ F1 below target. Consider more epochs or class weighting.")

print("\n=== Key Findings ===")
print("• DistilBERT successfully fine-tuned for 6-class medical query classification")
print("• Model ready to be used as routing layer in full RAG pipeline")
print("• Saved model location: models/classifier/distilbert_classifier/")

In [ ]:
# Save the fine-tuned model and tokenizer to models/classifier/distilbert_classifier/
import os
save_path = "../models/classifier/distilbert_classifier"
os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to: {save_path}")
print(f"   Files: {os.listdir(save_path)}")